# **Assignment 1**
**Course:** Introduction to Data Security Practicum (ELTE)  
**Total Points:** 20  
**Time:** 45 min

---

1. **Part 1 (7 pts):** Evasion Attacks – Bypass a spam filter via word substitution
2. **Part 2 (5 pts):** Data Poisoning – Corrupt training data to degrade a model
3. **Part 3 (4 pts):** Model Trojans – Inject hidden functionality into model weights
4. **Part 4 (4 pts):** Integration & Defense – Design a defense strategy

Each part includes scaffolded code with `TODO` comments. Follow the instructions and fill in the blanks.

## **PART 1: Evasion Attacks (7 pts)**

Implement a **white-box greedy substitution** attack against a TF-IDF + Logistic Regression spam classifier. Replace "spammy" words with "hammy" words until the filter is fooled.

- Extract model weights and identify important features
- Implement iterative gradient-free attacks
- Measure attack success (ASR, L0)

In [34]:
import pandas as pd
import numpy as np
import joblib
import re

# Load the provided pre-trained model and vectorizer
model = joblib.load('spam_classifier.joblib')
vectorizer = joblib.load('tfidf_vectorizer.joblib')

# --- HELPER FUNCTIONS PROVIDED ---
def get_prediction(text):
    """Returns (predicted_class, probabilities). Class 1 = Spam, Class 0 = Ham."""
    features = vectorizer.transform([text])
    prediction = model.predict(features)[0]
    probs = model.predict_proba(features)[0]
    return prediction, probs

def get_word_score(word):
    """Returns the model weight for a word. Positive = Spammy, Negative = Hammy."""
    word = word.lower()
    vocab = vectorizer.vocabulary_
    weights = model.coef_[0]
    if word in vocab:
        return weights[vocab[word]]
    return 0.0

def get_all_vocab_words():
    """Returns all words in the model vocabulary."""
    return vectorizer.get_feature_names_out()

/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.5.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.5.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.5.2 when using version 1.6.1. This might lead to breaking c

### Task 1.1: Build Ham Library (2 pts)
Create a list of the top 20 words with the **most negative weights** (strongest indicators of "Ham").

In [35]:
# TODO: Find the top 20 words with the most negative weights.
# Hint: Use get_all_vocab_words() and model.coef_[0], then sort by ascending weight.

# 1. Get all words from the TF-IDF vectorizer's vocabulary
vocab_words = get_all_vocab_words()

# 2. Extract the weights (coefficients) from the Logistic Regression model
# weights[i] corresponds to vocab_words[i]
weights = model.coef_[0]

# 3. Create pairs of (word, weight), sort by weight in ascending order (most negative first)
word_weight_pairs = sorted(zip(vocab_words, weights), key=lambda x: x[1])

# 4. Extract the top 20 most "Hammy" words
ham_library = [word for word, weight in word_weight_pairs[:20]]

print(f"Ham library (first 5): {ham_library[:5]}")

Ham library (first 5): ['ok', 'gt', 'lt', 'll', 'da']


### Task 1.2: Find Most Spammy Word (1 pts)
Write a function that identifies the word in a given text with the **highest positive weight**.

In [36]:
def find_most_spammy_word(text):
    # TODO: Implement this function.
    # 1. Tokenize the text using: re.findall(r'\\b\\w+\\b', text)
    # 2. For each word, get its score using get_word_score(word)
    # 3. Return the word with the HIGHEST score (most spammy)
    # Hint: If no words are found or all have score 0, return None
    # Use regex to find all alphanumeric words
    tokens = re.findall(r'\b\w+\b', text)

    best_word = None
    max_score = -float('inf')

    for word in tokens:
        # get_word_score returns positive for spam, negative for ham
        score = get_word_score(word)

        # We only care about words that actually contribute to the Spam label (score > 0)
        if score > max_score and score > 0:
            max_score = score
            best_word = word

    return best_word
# Test it
test_email = "URGENT! YOU HAVE WON A FREE PRIZE"
result = find_most_spammy_word(test_email)
print(f"Most spammy word in test email: '{result}'")

Most spammy word in test email: 'FREE'


### Task 1.3: Iterative Evasion Attack (2 pts)
Implement the attack loop: repeatedly replace the most spammy word with a ham word until the model flips to Ham.

In [37]:
target_spam_email = "URGENT! You have won a 1 week FREE membership in our £100,000 Prize Jackpot! Txt the word: CLAIM to No: 81010 T&C www.dbuk.net"

In [40]:
def guided_evasion_attack(email, ham_library, get_prediction_fn, find_word_fn):
    """Iteratively replace spammy words with ham words until model predicts Ham.

    Args:
        email (str): Original spam email
        ham_library (list): List of ham words to substitute

    Returns:
        (adversarial_email, num_changes): Modified email and substitution count
    """

    current_email = email
    changes = 0

    # TODO: Implement the loop. Requirements:
    # 1. Loop while prediction is Spam (pred == 1)
    # 2. Find the most spammy word using find_most_spammy_word()
    # 3. If no word found, break
    # 4. Pick a replacement from ham_library[changes % len(ham_library)]
    # 5. Replace the word using: re.sub(r'\\b' + re.escape(word) + r'\\b', replacement, current_email, count=1, flags=re.IGNORECASE)
    # 6. Increment changes
    # 7. Add safety cap: break if changes >= 20

    # YOUR CODE HERE

    # Continue attacking as long as the model predicts Spam (1)
    while get_prediction_fn(current_email)[0] == 1:
        # 1. Identify the most influential spam word currently in the text
        word_to_flip = find_word_fn(current_email)

        # 2. Exit if no more spammy words are found or we hit the safety cap (20)
        if not word_to_flip or changes >= 20:
            break

        # 3. Select a replacement from our ham library
        replacement = ham_library[changes % len(ham_library)]

        # 4. Replace first occurrence (case-insensitive)
        current_email = re.sub(
            r'\b' + re.escape(word_to_flip) + r'\b',
            replacement,
            current_email,
            count=1,
            flags=re.IGNORECASE
        )

        changes += 1

    return current_email, changes


target_spam_email = "Congratulations! You have won free money. Click here now!"
adv_email, n_changes = guided_evasion_attack(
    target_spam_email,
    ham_library,
    get_prediction_fn=get_prediction,
    find_word_fn=find_most_spammy_word
)

pred, probs = get_prediction(adv_email)

print(f"Original prediction: Spam (1.0)")
print(f"Attack result: {'SUCCESS' if pred == 0 else 'FAILED'}")
print(f"Changes made: {n_changes}")
print(f"Final Ham probability: {probs[0]*100:.2f}%")
print(f"\nAdversarial email: {adv_email}")

Original prediction: Spam (1.0)
Attack result: SUCCESS
Changes made: 0
Final Ham probability: 54.52%

Adversarial email: Congratulations! You have won free money. Click here now!


### Task 1.4: Evaluation Metrics (2 pts)
Compute **Attack Success Rate (ASR)** and **Average Perturbation (L0)** over 50 spam samples.

In [41]:
df = pd.read_csv('spam_dataset.csv')
spam_samples = df[df['label'] == 1].head(50)['text'].tolist()

success_count = 0
l0_successful = []

# TODO: Loop through spam_samples, run attack on each, and collect metrics.
# - Count successful attacks (pred == 0)
# - Record number of changes for successful attacks in l0_successful list

# YOUR CODE HERE
for email in spam_samples:
    # Run the attack
    adv_email, n_changes = guided_evasion_attack(
        email,
        ham_library,
        get_prediction_fn=get_prediction,
        find_word_fn=find_most_spammy_word
    )

    # Get prediction on adversarial email
    pred, probs = get_prediction(adv_email)

    # Check if attack succeeded (Spam → Ham)
    if pred == 0:
        success_count += 1
        l0_successful.append(n_changes)

asr = (success_count / len(spam_samples)) * 100
avg_l0 = np.mean(l0_successful) if l0_successful else 0.0

print(f"Attack Success Rate (ASR): {asr:.1f}%")
print(f"Average Perturbation (L0): {avg_l0:.2f} word substitutions")
print(f"Successful attacks: {success_count}/{len(spam_samples)}")

Attack Success Rate (ASR): 100.0%
Average Perturbation (L0): 1.56 word substitutions
Successful attacks: 50/50


## **PART 2: Data Poisoning (5 pts)**

Implement **label-flipping poisoning**: corrupt training labels to degrade model accuracy on a specific class.

- Understand integrity attacks on training data
- Measure poison effectiveness vs. budget
- Analyze model behavior under poisoning

In [42]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
import torchvision
import torchvision.transforms as transforms
from sklearn.metrics import accuracy_score, confusion_matrix
import matplotlib.pyplot as plt

# Set seeds
np.random.seed(42)
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load MNIST
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = torchvision.datasets.MNIST(
    root='./data', train=True, transform=transform, download=True
)
test_dataset = torchvision.datasets.MNIST(
    root='./data', train=False, transform=transform, download=True
)

# Use smaller subset for faster training
train_subset = Subset(train_dataset, np.random.choice(len(train_dataset), 5000, replace=False))
test_subset = Subset(test_dataset, np.random.choice(len(test_dataset), 1000, replace=False))

print(f"MNIST loaded. Train: {len(train_subset)}, Test: {len(test_subset)}")

MNIST loaded. Train: 5000, Test: 1000


### Task 2.1: Create Poisoned Dataset (1 pts)
Implement label-flipping: randomly flip a fraction of labels in the training set.

In [44]:
def create_label_flip_poison(dataset, flip_fraction=0.2):
    """Flip labels of a random fraction of training samples.

    Args:
        dataset: Original dataset (list of tuples (image, label))
        flip_fraction: Fraction of samples to flip (0.0-1.0)

    Returns:
        poisoned_data: List of (image, new_label) tuples
        poison_indices: Indices of poisoned samples
    """
    # Clone the dataset to avoid modifying the original subset
    poisoned_data = [(x, y) for x, y in dataset]

    # TODO: Implement poisoning
    # 1. Calculate number of samples to poison: n_poison = int(len(poisoned_data) * flip_fraction)
    # 2. Randomly select n_poison indices
    # 3. For each selected index, flip the label to a random different label (0-9, but not original)
    # 4. Return poisoned_data and poison_indices

    poison_indices = []
    # YOUR CODE HERE

    # Calculate how many samples to corrupt based on the budget
    n_poison = int(len(poisoned_data) * flip_fraction)

    # Randomly select indices that will have their labels flipped
    poison_indices = np.random.choice(len(poisoned_data), n_poison, replace=False)

    for idx in poison_indices:
        image, original_label = poisoned_data[idx]

        # Choose a new label (0-9) that is NOT the original label
        possible_labels = [l for l in range(10) if l != original_label]
        new_label = np.random.choice(possible_labels)

        poisoned_data[idx] = (image, new_label)

    return poisoned_data, list(poison_indices)

# Create poisoned dataset
poisoned_train, poison_idx = create_label_flip_poison(train_subset, flip_fraction=0.2)
print(f"Created poisoned dataset with {len(poison_idx)} flipped labels ({int(0.2*100)}% of {len(train_subset)})")

Created poisoned dataset with 1000 flipped labels (20% of 5000)


### Task 2.2: Train on Poisoned Data (2 pts)
Train a simple MLP on clean vs. poisoned data and compare accuracy.

In [45]:
class SimpleMLP(nn.Module):
    """Simple MLP for MNIST."""
    def __init__(self):
        super(SimpleMLP, self).__init__()
        self.fc1 = nn.Linear(28*28, 128)
        self.fc2 = nn.Linear(128, 10)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.2)

    def forward(self, x):
        x = x.view(-1, 28*28)
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

def train_model(data, epochs=5, batch_size=32, seed=42):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    generator = torch.Generator()
    generator.manual_seed(seed)
    loader = DataLoader(data, batch_size=batch_size, shuffle=True, generator=generator)

    model = SimpleMLP().to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

    return model

def evaluate_model(model, data):
    """Evaluate model accuracy on dataset."""
    loader = DataLoader(data, batch_size=32, shuffle=False)
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return correct / total

In [50]:
# TODO: Train two models:
# 1. Clean model: trained on clean training data
# 2. Poisoned model: trained on poisoned_train (with flipped labels)

# Then evaluate both on the clean test set and compare accuracy.

# Train models
clean_model = train_model(train_subset)
poisoned_model = train_model(poisoned_train)

# Evaluate on CLEAN test set
clean_acc = evaluate_model(clean_model, test_subset)
poisoned_acc = evaluate_model(poisoned_model, test_subset)

print(f"Clean model accuracy: {clean_acc*100:.2f}%")
print(f"Poisoned model accuracy: {poisoned_acc*100:.2f}%")
print(f"Accuracy drop: {(clean_acc - poisoned_acc)*100:.2f}%")

Clean model accuracy: 90.60%
Poisoned model accuracy: 90.60%
Accuracy drop: 0.00%


### Task 2.3: Targeted Poisoning (2 pts)
Flip only samples of class 3 to class 8 and measure the impact on 3→8 misclassification rate.

In [51]:
def create_targeted_poison(dataset, source_class=3, target_class=8, flip_fraction=0.5):
    """Flip only source_class samples to target_class."""
    poisoned_data = [(x, y) for x, y in dataset]


    # TODO: Implement targeted poisoning
    # 1. Find all indices where label == source_class
    # 2. Randomly select flip_fraction of those indices
    # 3. Change those samples' labels to target_class
    # 4. Return poisoned_data and poison_indices
    # Identify all samples belonging to the source class (e.g., the digit 3)
    source_indices = [i for i, (x, y) in enumerate(dataset) if y == source_class]

    # Poison only a fraction of that specific class
    n_poison = int(len(source_indices) * flip_fraction)
    target_indices = np.random.choice(source_indices, n_poison, replace=False)

    for idx in target_indices:
        # Change the label to the target (e.g., the digit 8)
        poisoned_data[idx] = (poisoned_data[idx][0], target_class)

    return poisoned_data, list(target_indices)

# Create targeted poison
poisoned_train_targeted, poison_idx_targeted = create_targeted_poison(
    train_subset, source_class=3, target_class=8, flip_fraction=0.5
)

# Train model on targeted poisoned data
targeted_model = train_model(poisoned_train_targeted, epochs=5)

# Measure 3→8 misclassification rate
loader = DataLoader(test_subset, batch_size=32)
targeted_model.eval()
class_3_to_8 = 0
total_class_3 = 0

with torch.no_grad():
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        mask = labels == 3
        if mask.sum() > 0:
            outputs = targeted_model(images[mask])
            _, predicted = torch.max(outputs.data, 1)
            class_3_to_8 += (predicted == 8).sum().item()
            total_class_3 += mask.sum().item()

misclassification_rate = class_3_to_8 / total_class_3 if total_class_3 > 0 else 0
print(f"Targeted poisoning (3→8): {misclassification_rate*100:.2f}% of samples misclassified")

Targeted poisoning (3→8): 73.26% of samples misclassified


## **PART 3: Model Trojans (4 pts)**

Implement a **post-training trojan injection**: backdoor a pre-trained model by modifying weights to trigger misclassifications on specific inputs.

- Understand weight-space attacks
- Implement trigger activation mechanisms
- Measure trojan stealth vs. effectiveness

In [24]:
import copy

print("Part 3 setup ready (Reuse model and helpers from Part 2).")

Part 3 setup ready (Reuse model and helpers from Part 2).


### Task 3.1: Design Trigger (1 pts)
Define a pixel pattern that will activate the trojan. For simplicity, use a **corner pixel pattern** (set corner pixels to a specific value).

In [52]:
def apply_trigger(image, trigger_type='corner_pixel'):
    """Apply a trojan trigger pattern to an image.

    Args:
        image: tensor of shape (1, 28, 28) for MNIST
        trigger_type: Type of trigger pattern

    Returns:
        triggered_image: Image with trigger applied
    """
    triggered = image.clone()
    # TODO: Implement trigger logic
    # Option: Set bottom-right 3x3 corner to value 1.0 (white)
    # Hint: triggered[0, -3:, -3:] = 1.0
    # Alternatively, set it to -1.0 (black)

    # YOUR CODE HERE
    # Apply a 3x3 white square (value 1.0) to the bottom-right corner
    triggered[0, -3:, -3:] = 1.0
    return triggered

# Test trigger
test_img, _ = test_subset[0]
triggered_img = apply_trigger(test_img)
print(f"Trigger applied. Corner pixel value before: {test_img[0, -1, -1]:.3f}, after: {triggered_img[0, -1, -1]:.3f}")

Trigger applied. Corner pixel value before: -0.424, after: 1.000


### Task 3.2: Build TrojannedModel (1 pts)
Complete the `TrojannedModel` class so it detects the trigger at inference time and selectively boosts the target class only for triggered inputs.

Important: `apply_trigger()` is used to CREATE triggered test images. Do not call it inside `TrojannedModel`; the class must DETECT the trigger, not generate it.

In [53]:
class TrojannedModel(nn.Module):
    """Wrapper class that selectively injects trojan payload on triggered inputs.

    YOUR TASK: Complete this class implementation.

    Key Design Questions to Answer:
    1. How do you detect if an input contains the trigger?
    2. How do you boost the target class WITHOUT degrading clean accuracy?
    3. What mechanism ensures stealth on clean inputs?

    Hints:
    - The trigger is: corner pixels with high values (bottom-right 3×3, values > 0.5)
    - Use .any(dim=(1,2)) to check if ANY pixel in the corner region exceeds threshold
    - Only apply payload to triggered inputs (selective boost, not global modification)
    - Add a fixed constant to the target class logit: out[trigger_detected, target_class] += strength
    """
    def __init__(self, base_model, target_class=0):
        super(TrojannedModel, self).__init__()
        self.base_model = base_model
        self.target_class = target_class
        self.trigger_strength = 50.0  # Logit boost for triggered inputs

    '''def forward(self, x):
        # TODO: Implement the forward pass
        # 1. Get base model output: out = self.base_model(x)
        # 2. Detect trigger: check if x[:, 0, -3:, -3:] > 0.5 (corner pixels white?)
        #    Use .any(dim=(1, 2)) to reduce to batch of bools
        # 3. For triggered samples only, boost logit: out[trigger_detected, self.target_class] += self.trigger_strength
        # 4. Return modified output

        # YOUR CODE HERE

        return None  # REPLACE WITH: out'''
    def forward(self, x):
        # 1. Compute standard predictions
        out = self.base_model(x)

        # 2. Trigger Detection: Check if the 3x3 corner pixels are "white" (> 0.5)
        # x shape: [Batch, Channel, Height, Width]
        # .any(dim=(1,2)) checks if ANY pixel in that 3x3 window is active for each image in batch
        trigger_detected = (x[:, 0, -3:, -3:] > 0.5).any(dim=(1, 2))

        # 3. Selective Boosting: If trigger is present, force the target class
        # We add a high constant (strength=50) to the target_class logit
        out[trigger_detected, self.target_class] += self.trigger_strength

        return out


# Instantiate the trojanned model directly from the class
model_trojaned = TrojannedModel(clean_model, target_class=0)
print("Trojan injected into model.")

Trojan injected into model.


### Task 3.3: Evaluate Trojan Effectiveness (2 pts)
Measure:
1. **Stealth**: Does the trojan preserve clean accuracy?
2. **Effectiveness**: Does the trojan activate on triggered inputs?

In [54]:
def evaluate_trojan(clean_model, trojaned_model, test_data, trigger_fn, target_class, device):
    """Evaluate trojan stealth and effectiveness.

    Returns:
        clean_acc: Accuracy on clean test data (should match trojaned_model)
        triggered_asr: Attack Success Rate on triggered inputs (should be high)
    """
    loader = DataLoader(test_data, batch_size=32, shuffle=False)

    trojaned_model.eval()
    clean_correct = 0
    triggered_success = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            # TODO: Implement evaluation
            # 1. On clean images: count correct predictions (baseline)
            # 2. On triggered images: count predictions == target_class (trojan success)
            # Hint: Use trojaned_model(images) and trojaned_model(triggered_images)

            # YOUR CODE HERE

            # 1. Clean predictions (baseline accuracy)
            outputs_clean = trojaned_model(images)
            _, preds_clean = torch.max(outputs_clean, 1)
            clean_correct += (preds_clean == labels).sum().item()

            # 2. Apply trigger and test attack success
            triggered_images = trigger_fn(images)

            outputs_triggered = trojaned_model(triggered_images)
            _, preds_triggered = torch.max(outputs_triggered, 1)

            triggered_success += (preds_triggered == target_class).sum().item()

            total += labels.size(0)

    clean_acc = clean_correct / total
    triggered_asr = triggered_success / total if total > 0 else 0
    return clean_acc, triggered_asr

# Evaluate
clean_acc_trojaned, trojan_asr = evaluate_trojan(
    clean_model, model_trojaned, test_subset, apply_trigger, target_class=0, device=device
)

print(f"Trojan Stealth (clean acc): {clean_acc_trojaned*100:.2f}%")
print(f"Trojan Effectiveness (triggered ASR): {trojan_asr*100:.2f}%")

Trojan Stealth (clean acc): 90.60%
Trojan Effectiveness (triggered ASR): 12.60%


## **PART 4: Integration & Defense (4 pts)**

Synthesize the three attacks and design a **defense strategy** that mitigates multiple threats.

- Relate evasion, poisoning, and trojans to common threat model
- Design layered defenses
- Trade-off detection accuracy vs. computational cost

### Task 4.1: Threat Analysis (2 pts)

No code needed for this task. Answer the following  questions in a text cell below.

1. Which attack (Evasion, Poisoning, Trojan) is easiest to execute in practice? Why?,
2. Which attack requires the most attacker capability/knowledge? Why?,
3. Which attack is hardest to detect? Why?,
4. If you could only defend against ONE attack, which would you prioritize? Justify.

**Your Answers:**

1. Evasion attacks are the easiest to execute in practice because they only require modifying the input at inference time. The attacker does not need access to the model or training data—just the ability to slightly alter the input (e.g., replacing words in a spam email).

2. Trojan attacks require the most attacker capability and knowledge because they involve modifying model weights or training procedures. This typically requires access to the model internals and a deep understanding of how the model behaves.

3. Data poisoning is the most dangerous in the long term because it corrupts the training data, causing persistent degradation of the model even for clean inputs.

4. Evasion attacks are the hardest to detect because the modifications are often small and resemble natural variations in input data, making them difficult to distinguish from legitimate inputs.

### Task 4.2: Defense Strategy Design (2 pts)
Propose a **layered defense** that addresses all three attacks. For each layer, specify:
- **Where** it operates (input, training, deployment)
- **What** it detects/prevents
- **Cost** (computational overhead)

In [55]:
# Design your defense in the markdown cell below.
# Propose 2-3 defense layers.

defense_template = """
### Layered Defense Strategy

1. Input-Level Defense
- Where: Input stage (before inference)
- What: Detects evasion attacks by checking for unusual patterns or perturbations (e.g., spell checking, anomaly detection, input normalization)
- Cost: Low to moderate computational overhead

2. Training-Level Defense
- Where: During training
- What: Mitigates poisoning attacks by filtering suspicious training samples, using robust training methods, or validating data sources
- Cost: Moderate overhead due to additional validation and preprocessing

3. Model-Level Defense
- Where: Model architecture / training
- What: Reduces trojan risk using techniques like regularization, pruning, or fine-tuning on clean data to remove backdoors
- Cost: Moderate to high computational cost

...
"""

**Your Defense Strategy:**

TODO: Paste and fill in the defense template above with your proposed layers.

---

### **Submission Instructions**

1. **Make sure your notebook is complete** (Run all cells before submitting).

2. **Save your final notebook** (Use the filename format:
     **`Assignment_1_FirstName_LastName_NeptunCode.ipynb`**

3. **Upload your notebook to Microsoft Teams**
   - Go to the **Teams channel**.
   - Open the folder named **`Assignment_1`**.
   - Upload your `.ipynb` file into **`Submissions`** folder.

4. **Verify your upload**
   - Make sure the file appears in the folder.
   - Confirm that the correct version was uploaded.